In [17]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

REPO_DIR = Path("/home/v-srsreenath/hari/claw/c-val").resolve()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

krew_bin = Path.home() / ".krew" / "bin"
if krew_bin.is_dir():
    path_parts = os.environ.get("PATH", "").split(os.pathsep)
    if str(krew_bin) not in path_parts:
        os.environ["PATH"] = str(krew_bin) + os.pathsep + os.environ.get("PATH", "")

from cval.k8s.discovery import discover_free_nodes, fully_free_node_names
from cval.storage.status import get_latest_status_rows, latest_status_rows_to_node_map
from cval.orchestrator.workflow import build_workflow_plan, workflow_plan_to_dict
from cval.jobs.manager import submit_workflow_plan, submission_result_to_dict
from cval.jobs.monitor import get_job_phases, monitor_jobs_until_terminal

CONFIG = {
    "namespace": "gcr-admin",
    "node_filter": "hgx",
    "days_threshold": 4,
    "batch_size": 3,
    "template_path": REPO_DIR / "ymls" / "specific-node-job.yml",
    "monitor_timeout_seconds": 180,
    "monitor_poll_interval_seconds": 30,
}

print(f"Using c-val package from: {REPO_DIR}")
print(f"Namespace: {CONFIG['namespace']}")
print("Notebook mode: read-only and dry-run unless explicit submit is run outside this notebook.")

Using c-val package from: /home/v-srsreenath/hari/claw/c-val
Namespace: gcr-admin
Notebook mode: read-only and dry-run unless explicit submit is run outside this notebook.


In [18]:
status_rows = get_latest_status_rows(namespace=CONFIG["namespace"])
db_status = latest_status_rows_to_node_map(status_rows)

nodes, totals = discover_free_nodes(node_name_filter=CONFIG["node_filter"])
free_nodes = fully_free_node_names(nodes)

print(f"Latest status rows: {len(status_rows)}")
print(f"Nodes with validation history: {len(db_status)}")
print(f"GPU nodes discovered: {len(nodes)}")
print(f"Fully free GPU nodes: {len(free_nodes)}")
print(f"GPU totals: {totals}")

Latest status rows: 104
Nodes with validation history: 104
GPU nodes discovered: 477
Fully free GPU nodes: 26
GPU totals: {'capacity': 3816, 'allocatable': 3816, 'used': 3524, 'free': 292}


In [19]:
LA = ZoneInfo("America/Los_Angeles")
validity_days = CONFIG["days_threshold"]
max_rows = 60

now_utc = datetime.now(timezone.utc)
summary_rows = []
for node, ts in sorted(db_status.items()):
    ts_dt_utc = datetime.fromtimestamp(int(ts), tz=timezone.utc)
    age_days = (now_utc - ts_dt_utc).total_seconds() / 86400.0
    summary_rows.append(
        {
            "node": node,
            "timestamp_ca": ts_dt_utc.astimezone(LA).strftime("%Y-%m-%d %H:%M:%S %Z"),
            "age_days": age_days,
            "validity": "valid" if age_days <= validity_days else "expired",
        }
    )

headers = ["node", "timestamp_ca", "age_days", "validity"]
formatted = [
    {
        "node": row["node"],
        "timestamp_ca": row["timestamp_ca"],
        "age_days": f"{row['age_days']:.2f}",
        "validity": row["validity"],
    }
    for row in summary_rows[:max_rows]
]
widths = {header: max([len(header), *(len(row[header]) for row in formatted)]) for header in headers}

print(f"Now (CA): {now_utc.astimezone(LA).strftime('%Y-%m-%d %H:%M:%S %Z')}")
print(f"Validity window: {validity_days} days")
print(f"Showing {len(formatted)} of {len(summary_rows)} nodes with history")
print("-" * (sum(widths[header] for header in headers) + 3 * (len(headers) - 1)))
print(" | ".join(header.ljust(widths[header]) for header in headers))
print("-" * (sum(widths[header] for header in headers) + 3 * (len(headers) - 1)))
for row in formatted:
    print(" | ".join(row[header].ljust(widths[header]) for header in headers))

Now (CA): 2026-06-10 16:10:09 PDT
Validity window: 4 days
Showing 60 of 104 nodes with history
-------------------------------------------------------------------
node                | timestamp_ca            | age_days | validity
-------------------------------------------------------------------
slc01-cl02-hgx-0003 | 2026-01-19 19:42:26 PST | 141.81   | expired 
slc01-cl02-hgx-0006 | 2026-01-23 12:18:26 PST | 138.12   | expired 
slc01-cl02-hgx-0008 | 2026-01-19 19:48:41 PST | 141.81   | expired 
slc01-cl02-hgx-0012 | 2026-01-19 19:42:26 PST | 141.81   | expired 
slc01-cl02-hgx-0013 | 2026-01-23 12:18:27 PST | 138.12   | expired 
slc01-cl02-hgx-0015 | 2026-01-21 10:23:02 PST | 140.20   | expired 
slc01-cl02-hgx-0021 | 2026-01-23 12:18:28 PST | 138.12   | expired 
slc01-cl02-hgx-0024 | 2026-01-19 19:48:42 PST | 141.81   | expired 
slc01-cl02-hgx-0029 | 2026-01-23 12:18:28 PST | 138.12   | expired 
slc01-cl02-hgx-0031 | 2026-01-19 19:48:43 PST | 141.81   | expired 
slc01-cl02-hgx-0033 |

In [20]:
plan = build_workflow_plan(
    free_nodes,
    db_status,
    days_threshold=CONFIG["days_threshold"],
    batch_size=CONFIG["batch_size"],
    template_path=CONFIG["template_path"],
)
plan_dict = workflow_plan_to_dict(plan)

print(f"Dry run: {plan_dict['dry_run']}")
print(f"Free nodes: {plan_dict['free_nodes_count']}")
print(f"Queue candidates: {plan_dict['queue_count']}")
print(f"Planned jobs: {len(plan_dict['planned_jobs'])}")
for job in plan_dict["planned_jobs"]:
    print(f"  priority={job['priority']} node={job['node']} reason={job['reason']} job={job['job_name']}")

Dry run: True
Free nodes: 26
Queue candidates: 26
Planned jobs: 3
  priority=1 node=slc01-cl02-hgx-0046 reason=never-tested job=hari-gcr-ceval-slc01-cl02-hgx-0046-1781133015
  priority=2 node=slc01-cl02-hgx-0064 reason=never-tested job=hari-gcr-ceval-slc01-cl02-hgx-0064-1781133015
  priority=3 node=slc01-cl02-hgx-0065 reason=never-tested job=hari-gcr-ceval-slc01-cl02-hgx-0065-1781133015


In [21]:
submission_preview = submit_workflow_plan(
    plan,
    namespace=CONFIG["namespace"],
    submit=False,
)
preview_dict = submission_result_to_dict(submission_preview)

print(json.dumps(preview_dict, indent=2))
print("No Kubernetes resources were submitted. Real submission requires the CLI path:")
print(
    "python -m cval.cli submit-plan --live-status "
    "--threshold-days 4 --batch-size 1 --submit --confirm submit"
)

{
  "namespace": "gcr-admin",
  "dry_run": true,
  "submitted_count": 0,
  "jobs": [
    {
      "node": "slc01-cl02-hgx-0046",
      "job_name": "hari-gcr-ceval-slc01-cl02-hgx-0046-1781133015",
      "action": "dry-run",
      "submitted": false,
      "stdout": ""
    },
    {
      "node": "slc01-cl02-hgx-0064",
      "job_name": "hari-gcr-ceval-slc01-cl02-hgx-0064-1781133015",
      "action": "dry-run",
      "submitted": false,
      "stdout": ""
    },
    {
      "node": "slc01-cl02-hgx-0065",
      "job_name": "hari-gcr-ceval-slc01-cl02-hgx-0065-1781133015",
      "action": "dry-run",
      "submitted": false,
      "stdout": ""
    }
  ]
}
No Kubernetes resources were submitted. Real submission requires the CLI path:
python -m cval.cli submit-plan --live-status --threshold-days 4 --batch-size 1 --submit --confirm submit
